<a href="https://colab.research.google.com/github/MatheusMacagnan/AulaPythonFaculdade/blob/Matheus_Alves_Macagnan/aula_11/Matheus_Macagnan_Aula_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Atividade 1

In [ ]:
# Bibliotecas nativas
import os
import glob
import xml.etree.ElementTree as ET

# Bibliotecas de terceiros
import pandas as pd

In [ ]:
import os

# Criando a pasta
os.makedirs("notas_xml", exist_ok=True)

# Dados das notas fiscais
notas_amostra = [
    {
        "num": "1001",
        "data": "2026-10-10",
        "fornecedor": "Tech Solutions LTDA",
        "cnpj": "12.345.678/0001-90",
        "total": "4500.00"
    },
    {
        "num": "1002",
        "data": "2026-10-12",
        "fornecedor": "Cloud Cloud Servicos",
        "cnpj": "98.765.432/0001-11",
        "total": "1280.50"
    },
    {
        "num": "1003",
        "data": "2026-10-15",
        "fornecedor": "Hardware & Redes SA",
        "cnpj": "45.123.890/0001-55",
        "total": "8900.00"
    }
]

# Criando os arquivos XML
for n in notas_amostra:

    conteudo = f"""<?xml version="1.0" encoding="UTF-8"?>
<NotaFiscal>
    <Numero>{n['num']}</Numero>
    <DataEmissao>{n['data']}T14:30:00-03:00</DataEmissao>
    <Emitente>
        <CNPJ>{n['cnpj']}</CNPJ>
        <Nome>{n['fornecedor']}</Nome>
    </Emitente>
    <ValorTotal>{n['total']}</ValorTotal>
</NotaFiscal>
"""

    with open(
        f"notas_xml/nfe_{n['num']}.xml",
        "w",
        encoding="utf-8"
    ) as arquivo:
        arquivo.write(conteudo)

print("Amostras criadas com sucesso!")

Amostras criadas com sucesso!


In [ ]:
# Lista para armazenar os dados extraídos
dados_notas = []

# Localizando todos os arquivos XML
arquivos_xml = glob.glob("notas_xml/*.xml")

# Percorrendo os arquivos encontrados
for arquivo in arquivos_xml:

    try:
        # Lendo o arquivo XML
        arvore = ET.parse(arquivo)
        raiz = arvore.getroot()

        # Extraindo as informações
        numero = raiz.findtext("Numero")
        data = raiz.findtext("DataEmissao")
        fornecedor = raiz.findtext("Emitente/Nome")
        cnpj = raiz.findtext("Emitente/CNPJ")
        total = raiz.findtext("ValorTotal")

        # Verificando se existem informações ausentes
        if not all([numero, data, fornecedor, cnpj, total]):
            raise ValueError("Existem informações ausentes na nota.")

        # Convertendo o valor para número
        total = float(total)

        # Armazenando os dados
        dados_notas.append({
            "Numero": numero,
            "Data de Emissao": data[:10],
            "Fornecedor": fornecedor,
            "CNPJ": cnpj,
            "Valor Total": total
        })

        print(f"Nota {numero} extraída com sucesso!")

    except (ET.ParseError, OSError, ValueError) as erro:
        print(f"Erro ao processar {arquivo}: {erro}")

print("\nExtração finalizada!")

Nota 1001 extraída com sucesso!
Erro ao processar notas_xml/nfe_1003.xml: not well-formed (invalid token): line 7, column 24
Nota 1002 extraída com sucesso!

Extração finalizada!


In [ ]:
# Criando o DataFrame
df = pd.DataFrame(dados_notas)

# Salvando os dados em Excel
with pd.ExcelWriter("notas_fiscais.xlsx", engine="openpyxl") as escritor:

    df.to_excel(
        escritor,
        sheet_name="Notas Fiscais",
        index=False
    )

    # Acessando a planilha
    planilha = escritor.sheets["Notas Fiscais"]

    # Formatando a coluna de valores
    for celula in planilha["E"][1:]:
        celula.number_format = 'R$ #,##0.00'

    # Ajustando a largura das colunas
    planilha.column_dimensions["A"].width = 15
    planilha.column_dimensions["B"].width = 20
    planilha.column_dimensions["C"].width = 30
    planilha.column_dimensions["D"].width = 25
    planilha.column_dimensions["E"].width = 20

print("Planilha Excel criada com sucesso!")

# Visualizando os dados
print(df)

Planilha Excel criada com sucesso!
  Numero Data de Emissao            Fornecedor                CNPJ  \
0   1001      2026-10-10   Tech Solutions LTDA  12.345.678/0001-90   
1   1002      2026-10-12  Cloud Cloud Servicos  98.765.432/0001-11   

   Valor Total  
0       4500.0  
1       1280.5  
